# 04b — EXP_02: RAGAS Evaluation (Claude Sonnet 4.6 judge — all 5 metrics)

Scores `results/exp_02_naive_rag__golden_234/predictions.jsonl` against the 234-row golden subset using the locked judge.

**Three families separation upheld**: generator = LLaMA · constructor = `gpt-4o` · judge = `claude-sonnet-4-6`.

## What gets scored

Unlike EXP_01 (No-RAG, where context-dependent metrics were `null` by design), EXP_02 retrieves real chunks → **all five RAGAS metrics apply**. The runner auto-detects this via the `_has_context` flag in the dataset rows.

| Metric | Run for EXP_02? | What it measures |
|---|---|---|
| Faithfulness | ✓ | Is each claim in the answer grounded in the retrieved context? |
| Context Precision | ✓ | Of the retrieved chunks, how many are *relevant* to the question? |
| Context Recall | ✓ | Of the reference answer's claims, how many are supported by retrieved chunks? |
| Answer Relevancy | ✓ | Does the answer address the question? |
| Answer Correctness | ✓ | Does the answer match the reference? |

Plus the derived `RAGAS_Hallucination_Rate` = fraction of rows with `Faithfulness < 0.5` per [`docs/todo.md` §5 EXP_01](../docs/todo.md).

## Three gated stages — same pattern as 04a

| Stage | Surface | Wall time | Cost (Claude Sonnet 4.6) |
|---|---|---|---|
| **A** Smoke | 10 rows × 5 metrics | ~1 min | **~$1.50** |
| **B** Full | 234 rows × 5 metrics | ~10–20 min | **~$30–40** |
| **C** NaN rescore | only NaN rows after Stage B | ~1–3 min | **~$1–3** if needed |

**Critical**: this is ~3× the cost of EXP_01's RAGAS pass because all five metrics fire (vs only 2 for No-RAG). The `RunConfig(max_workers=4, max_retries=10, max_wait=120)` resilience layer is on by default — expect on-first-pass NaN rate < 5 %.

Stage C only fires when `RUN_RESCORE_NANS = True`. Once `ragas_scores.csv` exists, the runner skips judging on re-runs (file-level resumability).

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.eval.ragas_eval import score_predictions

for k in ["ANTHROPIC_API_KEY"]:
    print(f"{k}: {'\u2713 present' if os.environ.get(k) else '\u2717 MISSING — add to .env at repo root and reload'}")

print("Repo root:", REPO_ROOT)

## 2. Configuration

In [ ]:
PRED_DIR = REPO_ROOT / "results" / "exp_02_naive_rag__golden_234"
GOLDEN_PATH = REPO_ROOT / "data" / "processed" / "golden_ragas_300.jsonl"
CHUNKS_PATH = REPO_ROOT / "data" / "processed" / "chunks.parquet"

RUN_SMOKE = True
SMOKE_N = 10
RUN_FULL = False           # flip to True only after smoke looks clean
RUN_RESCORE_NANS = False   # flip to True if Stage B leaves residual NaN cells
OVERWRITE_SCORES = False   # True forces a fresh judge pass; mutually exclusive with RUN_RESCORE_NANS

print("Predictions:", PRED_DIR)
print("Golden     :", GOLDEN_PATH)
print("Chunks     :", CHUNKS_PATH)

assert PRED_DIR.exists(), (
    f"{PRED_DIR} doesn't exist — run notebooks/04b_exp02_naive_rag.ipynb Stage B first."
)

## 3. Stage A — Smoke (10 rows × 5 metrics · ~1 min · ~$1.50)

Smoke writes to a separate scratch path so the canonical 234-row CSV isn't polluted with a partial 10-row run.

In [ ]:
import pandas as pd
import shutil

if RUN_SMOKE:
    smoke_dir = PRED_DIR.parent / "exp_02_naive_rag__golden_234_ragas_smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    for fname in ("predictions.jsonl", "retrieval.jsonl", "summary.json"):
        shutil.copy2(PRED_DIR / fname, smoke_dir / fname)

    smoke_summary = score_predictions(
        predictions_dir=smoke_dir,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=SMOKE_N,
        overwrite=OVERWRITE_SCORES,
    )
    print("\n=== Stage A smoke summary (10 rows × 5 metrics) ===")
    for k, v in smoke_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")):
            print(f"  {k:30s} : {v}")

    smoke_scores = pd.read_csv(smoke_dir / "ragas_scores.csv")
    print("\n=== Per-row scores (smoke 10) ===")
    cols_to_show = [
        c for c in smoke_scores.columns
        if c in ("question_id", "_is_correct",
                 "faithfulness", "context_precision", "context_recall",
                 "answer_relevancy", "answer_correctness")
    ]
    print(smoke_scores[cols_to_show].to_string(index=False))
else:
    print("RUN_SMOKE = False — skipped.")

**Acceptance gates (Stage A)** — flip `RUN_FULL = True` only when all five pass:

1. **No errors thrown** by the `evaluate()` call.
2. **All five metric columns exist** in the scores CSV: `faithfulness`, `context_precision`, `context_recall`, `answer_relevancy`, `answer_correctness`.
3. **NaN rate < 30 %** on the smoke. The new resilience layer should keep this very low; if it's high on smoke, do NOT scale.
4. **Faithfulness > 0** on at least some rows. The whole point of EXP_02 is that retrieval enables a non-zero Faithfulness baseline (vs EXP_01's `null`).
5. **Sane Answer Correctness correlation** — rows where `_is_correct=True` should average higher than `_is_correct=False`.

## 4. Stage B — Full 234 (~10–20 min · ~$30–40)

Writes to the canonical `results/exp_02_naive_rag__golden_234/`. Updates `summary.json` in place. The 5-metric judge pass is the most expensive single step in Phase 4 — this same pattern repeats 4× more times for EXP_03/04/05.

In [ ]:
if RUN_FULL:
    full_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=None,  # full 234
        overwrite=OVERWRITE_SCORES,
    )
    print("\n=== Stage B full summary (234 rows × 5 metrics) ===")
    for k, v in full_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")) or k == "Acuuracy":
            print(f"  {k:30s} : {v}")
else:
    print("RUN_FULL = False — set to True after Stage A passes its gates.")

## 5. Stage C — NaN rescore (re-judge transient-failure rows)

If Stage B leaves residual NaN cells from any transient API failures, this stage re-judges only those rows and merges new scores into the existing CSV. With the `RunConfig(max_workers=4, ...)` resilience layer the residual NaN rate should be small; this stage is the cheap recovery path if not.

**Cost** — only the NaN rows pay. If Stage B comes back clean, set `RUN_RESCORE_NANS = False` and skip.

In [ ]:
if RUN_RESCORE_NANS:
    rescored_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        rescore_nans=True,
    )
    print("\n=== After NaN rescore ===")
    for k, v in rescored_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")):
            print(f"  {k:30s} : {v}")
    df = pd.read_csv(PRED_DIR / "ragas_scores.csv")
    for col in ("faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness"):
        if col in df.columns:
            n_nan = pd.to_numeric(df[col], errors="coerce").isna().sum()
            print(f"  {col:25s}: {n_nan} NaN remaining of {len(df)}")
else:
    print("RUN_RESCORE_NANS = False — skip if Stage B came back clean.")

## 6. Inspect — RAGAS lifts vs EXP_01

The headline question: does retrieval improve Answer Correctness vs the No-RAG baseline (EXP_01 had AC = 0.8738 over n=137)? And what's the Faithfulness baseline (which EXP_01 couldn't measure)?

In [ ]:
scores_path = PRED_DIR / "ragas_scores.csv"
if scores_path.exists():
    df = pd.read_csv(scores_path)
    print(f"\nrows: {len(df)}")
    for col in ("faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness"):
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        print(f"\n  {col} distribution:")
        print(f"    mean={s.mean():.4f}  median={s.median():.4f}  std={s.std():.4f}  NaN={s.isna().sum()}")
        if col == "faithfulness":
            hallucination_rate = (s < 0.5).sum() / s.notna().sum() if s.notna().sum() else float('nan')
            print(f"    Hallucination rate (Faithfulness < 0.5): {hallucination_rate:.4f}")
        print("    by _is_correct:")
        print(df.groupby("_is_correct")[col].agg(["mean", "count"]).round(4))
else:
    print("No ragas_scores.csv yet — run Stage B first.")

---

**Next.** With EXP_02's full RAGAS row complete, the next architectures follow the same template:

- **EXP_03 Sparse RAG** — build `src/retrieval/sparse.py` (BM25 wrapper around the existing `bm25_top_k` helper) and replicate this pair of notebooks (`04c_exp03_sparse_rag.ipynb` + `04c_exp03_ragas.ipynb`).
- **EXP_04 Hybrid RAG** — refactor `src/retrieval/hybrid.py` under the `Retriever` ABC and replicate again.
- **EXP_05 Multi-Hop RAG** — build `src/retrieval/multi_hop.py` (decompose → 1–3 hops → accumulate) and replicate; this one will take longer per question due to the iterative retrieval.